In [0]:
# replace with your catalog
CATALOG = spark.catalog.currentCatalog()
CATALOG = "nikkthegreek"

In [0]:
import os
import sys
import platform
from lakehouse.spark import bronze
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

In [0]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

In [0]:
try:
    spark
except NameError:
    builder = (
        SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
        .master("local[4]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 1. Set Up

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

In [0]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Stream


In [0]:
path = f"D:/Data/{CATALOG}/streamdata/"
df_json_1 = spark.createDataFrame(
    [
        (100, "Hyukjin Kwon1"),
    ],
    ["age", "name"],
)
df_json_1.coalesce(1).write.mode("overwrite").format("json").save(path)

In [0]:
class TestStream(bronze.Bronze):
    def custom_load(self, table):
        df = spark.readStream.schema("age BIGINT, name STRING").json(path)
        return df

    def checkpoint_path(self, table):
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}/checkpoint"


instance = TestStream(spark, **options)

In [0]:
(instance.load().transform().write(mode="stream").execute("stream"))
spark.sql(f"SELECT * FROM {CATALOG}.bronze.stream").show(truncate=False)

# 6 Clean Up

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
import shutil

shutil.rmtree(f"D:/Data/{CATALOG}")
#spark.stop()